In [ ]:
import copy
import tqdm
import numpy as np
import os
!pip install textgrid
import textgrid
import itertools
import multiprocessing
import time
from joblib import Parallel, delayed

In [5]:
#(this code for windows only, otherwise you could try colab version)


# set the R path :
# You may need to change the filepath based on your installation
os.environ['R_HOME'] = 'C:\\Program Files\\R\\R-4.4.1'
os.environ["PATH"] += os.pathsep + r"C:\Program Files\R\R-4.4.1\bin\x64"

from rpy2.robjects.packages import importr
from rpy2.robjects import Formula, pandas2ri
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

# Activate the pandas conversion
pandas2ri.activate()


# First run, please install below packages:
#utils = importr('utils')
#utils.install_packages('lme4')


### load human data from xlsx file

In [22]:
human_result_path=r"..\data\test.xlsx"
human_result = pd.read_excel(human_result_path)
human_result_1a=human_result[human_result["Experiment"]=="1a"]

### load audio file path

In [6]:
def get_pathset(paths):
    return [os.path.join(dir, each_file) for dir, mid, files in os.walk(paths) for each_file in files if each_file.endswith(".wav")]
audio_dir =r"..\data\speech_files"
set1_list=[0,1,2,3,4,5,6,7,8,9,10,12,13,14,15,16]
set2_list=[17,18,19,20,21,22,24,25,26,27,28,29,30,31,37,40]

### load 3d-tSNE representations from pkl file

In [14]:
import pickle
with open("..\data\hubert_words_Transformer.pkl", "rb") as file:
    tSNE_representations = pickle.load(file)

In [ ]:
def get_keywords_dict(human_result_1a):
    '''
    Return a dict, key: sentenceID, values: keywords
    '''
    keywords_dict={}
    for each_ in human_result_1a.values:
        sentenceID=each_[human_result_1a.columns.get_loc("SentenceID")]
        if sentenceID not in keywords_dict:
            keywords_dict[sentenceID]=[]
        keyword=each_[human_result_1a.columns.get_loc("Keyword")]
        if keyword not in keywords_dict[sentenceID]:
            keywords_dict[sentenceID].append(keyword)
    return dict(sorted(keywords_dict.items()))

def create_set(audio_dir, df, reduced_data):
    """
    This function is for pairing 3d-tSNE, audio_data and human_data. 
    """
    set1_list=[0,1,2,3,4,5,6,7,8,9,10,12,13,14,15,16]
    set2_list=[17,18,19,20,21,22,24,25,26,27,28,29,30,31,37,40]
    keywords_dict=get_keywords_dict(df)
    keywords=[j for i in list(keywords_dict.values()) for j in i]
    audio_path=get_pathset(audio_dir)[::-1]
    out_dict={}
    word_features=[[] for i in range(len(keywords))]
    for __, each_path in enumerate(audio_path):
        current_talker=os.path.basename(each_path)[:13]
        if current_talker not in out_dict.keys():
            out_dict[current_talker]=[[] for i in range(32)]
        tg = textgrid.TextGrid.fromFile(each_path[:-3]+"TextGrid")
        tg_sentence = tg[0]
        for _,i in enumerate(tg[0]):
            if i.mark!="":
                tg_sentence[_-1].maxTime=tg_sentence[_].minTime
        tg_sentence = [i for i in tg_sentence if i.mark!=""]
        tg_sentence=[tg_sentence[i] for i in set1_list+set2_list]
        tg_word = [i for i in tg[1] if i.mark!="" and i.mark!="sp"]
        count=0
        for _,each_sentence in enumerate(tg_sentence):
            sentence_total_length=each_sentence.maxTime-each_sentence.minTime
            for key_word in list(keywords_dict.values())[_]:

                for each_word_tg in tg_word:
                    if each_word_tg.mark.lower()==key_word:
                        if each_word_tg.minTime >= each_sentence.minTime and each_word_tg.maxTime <= each_sentence.maxTime:
                            start=each_word_tg.minTime
                            end=each_word_tg.maxTime
                            break

                word_cut_start=start-each_sentence.minTime
                word_cut_end=end-each_sentence.minTime
                word_start=round(reduced_data[_][__].shape[0]*word_cut_start/sentence_total_length)
                word_end=round(reduced_data[_][__].shape[0]*word_cut_end/sentence_total_length)
                features=copy.deepcopy(reduced_data[_][__][word_start:word_end,:])
                word_features[count].append(features)
                out_dict[current_talker][_].append(features)
                count+=1
    return word_features,out_dict
word_features,out_dict=create_set(audio_dir, human_result_1a, tSNE_representations)

In [ ]:
from numba import njit
@njit
def weighted_minkowski(vec1, vec2,  tau, w=1):

    total = 0.0
    for m in range(len(vec1)):
        diff = w*abs(vec1[m] - vec2[m])
        total += (diff ** tau)
    return total**(1/tau)#np.sqrt(total)

@njit
def dtw_sim(seq1, seq2,  tau, k):
    n, m = len(seq1), len(seq2)
    dtw_matrix = np.full((n+1, m+1), np.inf)
    dtw_matrix[0, 0] = 0.0

    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = weighted_minkowski(seq1[i-1], seq2[j-1], tau)
            dtw_matrix[i, j] = cost + min(dtw_matrix[i-1, j],    # insertion
                                         dtw_matrix[i, j-1],    # deletion
                                         dtw_matrix[i-1, j-1])  # match
    return np.exp(-(dtw_matrix[n, m]/((n+m)/2))*k)# change to *k,


In [19]:
def get_training_paths(TrainingTalkerID):
    TalkerID=[]
    for each_ID in TrainingTalkerID.split(", "):
        if each_ID[:3]=="CMN":
            TalkerID.append(f"ALL_{each_ID[-3:]}_M_CMN")
        else:
            TalkerID.append(f"ALL_{each_ID[-3:]}_M_ENG")
    return TalkerID

def get_keywords_list(df):
    '''
    build a dictionary of list, output like this:
    {
        'HT1_S001': ['boy', 'fell', 'window'],
        'HT1_S002': ['wife', 'helped', 'husband'],
        'HT1_S003': ['big', 'dogs', 'be', 'dangerous'],
    }
    '''
    out_dict={}
    for each_ in human_result_1a.values:
        keyword_loc=df.columns.get_loc("Keyword")
        key_word = each_[keyword_loc]
        sentenceID = each_[df.columns.get_loc("SentenceID")]
        if sentenceID not in out_dict.keys():
            out_dict[sentenceID]=[]
        if key_word not in out_dict[sentenceID]:
            out_dict[sentenceID].append(key_word)
    return out_dict

def get_exposure_set(feature_dict,trainingTalkerID,sentenceID,key_word):
    # load the exposure representations
    keywors_list = get_keywords_list(human_result_1a)
    set1_list=[0,1,2,3,4,5,6,7,8,9,10,12,13,14,15,16]
    set2_list=[17,18,19,20,21,22,24,25,26,27,28,29,30,31,37,40]
    features=[]
    for i in trainingTalkerID:
        sentence_ind=(set1_list+set2_list).index(int(sentenceID[-3:])-1)
        key_word_ind=keywors_list[sentenceID].index(key_word)
        features.append(copy.deepcopy(feature_dict[i][sentence_ind][key_word_ind]))
    return features

def get_test_feature(feature_dict, test_talker, sentenceID, key_word):
    # load the test representations
    keywors_list = get_keywords_list(human_result_1a)
    set1_list=[0,1,2,3,4,5,6,7,8,9,10,12,13,14,15,16]
    set2_list=[17,18,19,20,21,22,24,25,26,27,28,29,30,31,37,40]
    sentence_ind=(set1_list+set2_list).index(int(sentenceID[-3:])-1)
    key_word_ind=keywors_list[sentenceID].index(key_word)
    return copy.deepcopy(feature_dict[test_talker][sentence_ind][key_word_ind])


def sim_measure1(df, feature_dict,tau, k):
    '''
    This function is for computing the similarity values for each row of human data.
    '''
    
    train_set_dict={}
    test_word_dict={}
    # set the output dataframe
    out_df=pd.DataFrame(columns=['Condition2', 'TrainingTalkerID', 'TestTalkerID','SentenceID', 'Keyword',  'distance_min', 'IsCorrect'])#'trial',
    for each_ in tqdm.tqdm(df.values):
        #load some colnames or filenames
        keyword_loc=df.columns.get_loc("Keyword")
        sentence_loc= df.columns.get_loc("SentenceID")
        training_talker_loc=df.columns.get_loc("TrainingTalkerID")
        test_file = [os.path.basename(each_[df.columns.get_loc("Filename")])[:13]]
        key_word = each_[keyword_loc] 
        TrainingTalkerID = each_[training_talker_loc] 
        sentenceID = each_[df.columns.get_loc("SentenceID")]
        trainingTalkerID=get_training_paths(TrainingTalkerID)
        train_talker_key=",".join(sorted(trainingTalkerID))
        
        #load exposure data
        if train_talker_key not in train_set_dict:
            train_set_dict[train_talker_key]={}
        if sentenceID not in train_set_dict[train_talker_key]:
            train_set_dict[train_talker_key][sentenceID]={}
        if key_word not in train_set_dict[train_talker_key][sentenceID]:

            training_features=get_exposure_set(feature_dict,trainingTalkerID,sentenceID,key_word)
            train_set_dict[train_talker_key][sentenceID][key_word]=copy.deepcopy(training_features)
        else:
            training_features=train_set_dict[train_talker_key][sentenceID][key_word]

        #load test data
        if test_file[0] not in test_word_dict:
            test_word_dict[test_file[0]]={}
        if sentenceID not in test_word_dict[test_file[0]]:
            test_word_dict[test_file[0]][sentenceID]={}
        if key_word not in test_word_dict[test_file[0]][sentenceID]:
            test_feature = get_test_feature(feature_dict, test_file[0], sentenceID, key_word)
            test_word_dict[test_file[0]][sentenceID][key_word]=copy.deepcopy(test_feature)
        else:
            test_feature = test_word_dict[test_file[0]][sentenceID][key_word]
            
        # compute the similarity
        sims=[]
        for _, each_train_feature in enumerate(training_features):
            X=each_train_feature
            Y=test_feature
            normalized_sim = dtw_sim(X, Y, tau, k)
            sims.append(normalized_sim)
        out_df.loc[len(out_df)]=[each_[df.columns.get_loc("Condition2")], each_[df.columns.get_loc("TrainingTalkerID")],
                                    each_[df.columns.get_loc("TestTalkerID")], each_[sentence_loc],
                                    each_[df.columns.get_loc("Keyword")],
                                    np.max(sims),#pick the max sim value
                                    each_[df.columns.get_loc("IsCorrect")]]
    return out_df

In [20]:
def objective_function(params, out_dict, human_result_1a):
    """
    compute the z-value based on our select parameters
    """
    tau, k = params
    print(tau, k)
    out_df=sim_measure1(human_result_1a, out_dict,tau, k)
    new_df=copy.deepcopy(out_df)
    new_df['similarity'] = new_df['distance_min'].apply(lambda x:np.max(x))
    
    TTID=new_df.columns.get_loc("TrainingTalkerID")
    new_TrainingTalkerID=[]
    for i in new_df.values:
        new_TrainingTalkerID.append( ",".join(sorted(i[TTID].split(", "))) )
    new_df["TrainingTalkerID1"]=new_TrainingTalkerID
    new_df['trial'] = (new_df.groupby(['Keyword', 'Condition2', 'TrainingTalkerID1', 'TestTalkerID', 'SentenceID'], as_index=False).ngroup() + 1)


    new_df=new_df.groupby(['Keyword', 'Condition2', 'TrainingTalkerID1', 'TestTalkerID', 'SentenceID','trial'], as_index=False).agg( #'sentenceID'
        IsCorrect=('IsCorrect', 'mean'),
        #distance=('distance_min', 'min'),
        similarity=('similarity', 'mean'),
        numCorrect=('IsCorrect', lambda x: (x==1).sum()),
        numIncorrect=('IsCorrect', lambda x: (x==0).sum())
        )
    new_df['similarity_scaled'] = (new_df['similarity'] - np.mean(new_df['similarity'])) / (2 * np.std(new_df['similarity']))
    if np.std(new_df['similarity']) < 0.01:
        return {
                    "tau": [tau], 
                    "k": [k], 
                    "z-value": [0],  
                    "mean_sim": [float(np.mean(new_df["similarity"]))], 
                    "sd_sim": [float(np.std(new_df["similarity"]))],  
                    "error": ["std_sim<0.01"]  # 
                }
    else:
        try:
            import rpy2.robjects as ro
            import rpy2
            from rpy2.robjects import pandas2ri
            
            pandas2ri.activate()
            base = importr('base')
            stats = importr('stats')
            lme4 = importr('lme4')
            ro.r('options(warn=2)')
            #r_data = ro.conversion.py2rpy(new_df)
            r_data = pandas2ri.py2rpy(new_df)
            formula = Formula('cbind(numCorrect, numIncorrect) ~ 1 + similarity_scaled + (1 | SentenceID / Keyword) + (1| TestTalkerID)')
            glmerControl = lme4.glmerControl(optimizer="bobyqa", optCtrl=ro.vectors.ListVector({'maxfun': 1e6}))
        except Exception as e:
            return {
                    "tau": [tau], 
                    "k": [k], 
                    "z-value": [0],  
                    "mean_sim": [float(np.mean(new_df["similarity"]))], 
                    "sd_sim": [float(np.std(new_df["similarity"]))],  
                    "error": [e]  # 
                }
        try:
            model = lme4.glmer(formula, data=r_data, control=glmerControl, family=stats.binomial(link="logit"))
            #log_likelihood = ro.r['logLik'](model)
            summary = base.summary(model)
            coefficients = summary.rx2('coefficients')
            #z_value = coefficients.rx('similarity_scaled', 'z value')[0]
            z_value=coefficients[1][2]
            return {
                    "tau": [tau], 
                    "k": [k], 
                    "z-value": [z_value],  
                    "mean_sim": [float(np.mean(new_df["similarity"]))], 
                    "sd_sim": [float(np.std(new_df["similarity"]))],  
                    "error": ["NA"]  # 
                }
        except Exception as e:
            return {
                    "tau": [tau], 
                    "k": [k], 
                    "z-value": [0],  
                    "mean_sim": [float(np.mean(new_df["similarity"]))], 
                    "sd_sim": [float(np.std(new_df["similarity"]))],  
                    "error": [str(e)]  # 
                }

In [ ]:
#change the tau and k range here:
tau_values = 10**np.arange(-1,1,0.25)
k_values = 10**np.arange(-4,2.2,0.2)

pandas2ri.activate()
params_grid=list(itertools.product(tau_values, k_values))

start_time = time.time()
n_jobs = min(32, multiprocessing.cpu_count()) # you could change cpu number here
results = Parallel(n_jobs=n_jobs)(delayed(objective_function)(_,out_dict, human_result_1a) for _ in params_grid)
end_time = time.time()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import griddata

def plot_gridsearch_result(results):
    df = pd.json_normalize(results)
    df['tau'] = df['tau'].apply(lambda x: x[0] if isinstance(x, list) else x)
    df['k'] = df['k'].apply(lambda x: x[0] if isinstance(x, list) else x)
    df['z-value'] = df['z-value'].apply(lambda x: x[0] if isinstance(x, list) else x)
    df['mean_sim'] = df['mean_sim'].apply(lambda x: x[0] if isinstance(x, list) else x)
    df['sd_sim'] = df['sd_sim'].apply(lambda x: x[0] if isinstance(x, list) else x)
    tau_vals = tau_values  
    k_vals = k_values
    tau_grid, k_grid = np.meshgrid(tau_vals, k_vals)  
    fig = make_subplots(
        rows=1, cols=3, 
        subplot_titles=['z-value', 'mean_sim', 'sd_sim'], 
        specs=[[{'type': 'surface'}, {'type': 'surface'}, {'type': 'surface'}]]
    )
    z_grid = griddata((df['tau'], df['k']), df['z-value'], (tau_grid, k_grid), method='nearest')
    mean_sim_grid = griddata((df['tau'], df['k']), df['mean_sim'], (tau_grid, k_grid), method='nearest')
    sd_sim_grid = griddata((df['tau'], df['k']), df['sd_sim'], (tau_grid, k_grid), method='nearest')


    fig.add_trace(go.Surface(z=z_grid, x=tau_grid, y=k_grid, colorscale='RdYlGn', showscale=False), row=1, col=1)
    fig.add_trace(go.Surface(z=mean_sim_grid, x=tau_grid, y=k_grid, colorscale='RdYlGn', showscale=False), row=1, col=2)
    fig.add_trace(go.Surface(z=sd_sim_grid, x=tau_grid, y=k_grid, colorscale='RdYlGn', showscale=False), row=1, col=3)


    fig.update_layout(
        width=1700, 
        height=600,  
        title='3D Surface Plots for z-value, mean_sim, and sd_sim from LSR of 24th-Transformer-layer (Xie 2021)',
        scene=dict(
            xaxis_title='Tau',
            yaxis_title='K',
            zaxis_title='Z-Value',
            yaxis=dict(
                type='log',
                tickvals=[1e-4, 1e-3, 1e-2, 1e-1, 1, 10], 
                ticktext=['1e-4', '1e-3', '1e-2', '1e-1', '1', '10'],
            )
        ),
        scene2=dict(
            xaxis_title='Tau',
            yaxis_title='K',
            zaxis_title='Mean-Sim',
            yaxis=dict(
                type='log',
                tickvals=[1e-4, 1e-3, 1e-2, 1e-1, 1, 10], 
                ticktext=['1e-4', '1e-3', '1e-2', '1e-1', '1', '10'],
            )
        ),
        scene3=dict(
            xaxis_title='Tau',
            yaxis_title='K',
            zaxis_title='SD-Sim',
            yaxis=dict(
                type='log',
                tickvals=[1e-4, 1e-3, 1e-2, 1e-1, 1, 10], 
                ticktext=['1e-4', '1e-3', '1e-2', '1e-1', '1', '10'],
            ),
            zaxis=dict(
                range=[0,0.5]
            )
        ),
        showlegend=False,
        coloraxis_showscale=False
        
    )
    fig.show()
plot_gridsearch_result(results)

In [98]:

import pickle
with open("..\O_Jan31\hubert_words_CNN.pkl", "rb") as file:
    sentence_matrix2 = pickle.load(file)

word_features2,out_dict2=create_set(audio_dir, human_result_1a, sentence_matrix2)
tau_values = np.arange(1,11,1)
k_values = 10**np.arange(-4,1.1,0.1)
import itertools
import multiprocessing
import time
from joblib import Parallel, delayed
pandas2ri.activate()
params_grid=list(itertools.product(tau_values, k_values))

start_time = time.time()
n_jobs = min(32, multiprocessing.cpu_count())
results2 = Parallel(n_jobs=n_jobs)(delayed(objective_function)(_,out_dict2, human_result_1a) for _ in params_grid)
end_time = time.time()
print(f"Total wall time: {end_time - start_time:.2f} seconds")

Total wall time: 2084.34 seconds
